# 02 — Chunking Strategies

Compare different text chunking approaches and measure their impact on retrieval quality.

## What you'll learn
- Fixed-size vs recursive vs semantic chunking
- How chunk size and overlap affect results
- Measuring retrieval quality across strategies
- Visualizing chunk boundaries

In [ ]:
# !pip install langchain langchain-openai langchain-community matplotlib numpy

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Ready.")

## 1. Sample Document

A longer text to demonstrate how different chunking strategies break it up.

In [ ]:
sample_document = """Retrieval-Augmented Generation (RAG) is a technique that enhances large language models
by combining them with external knowledge retrieval. The core idea is simple: instead of asking
the model to generate answers purely from its training data, you first retrieve relevant documents
from a knowledge base and include them in the prompt. This grounds the model's response in real,
factual sources.

The RAG pipeline has several key components. First, documents are loaded from various sources such
as PDFs, databases, or web pages. These documents are then split into smaller chunks. Each chunk
is converted into a numerical vector using an embedding model. These vectors are stored in a
vector database like Pinecone, Chroma, or Weaviate.

When a user asks a question, the question is converted to a vector using the same embedding model.
The vector database performs a similarity search to find the most relevant chunks. These chunks are
retrieved and passed to a language model along with the original question. The language model
generates an answer based on the provided context.

Chunking is critical to RAG performance. If chunks are too large, retrieval becomes imprecise
because the relevant information is buried in irrelevant content. If chunks are too small, you
lose important context. The chunking strategy determines how text is divided. Fixed-size chunking
splits text into equal-sized pieces. Recursive splitting tries to respect natural boundaries
like paragraphs and sentences. Semantic chunking groups sentences by meaning.

Overlap between chunks helps maintain continuity. When a concept spans the boundary between two
chunks, overlap ensures both chunks contain enough context. However, too much overlap increases
storage costs and retrieval latency. A common starting point is 10-20% overlap.

Embedding quality directly impacts retrieval accuracy. Better embeddings capture deeper semantic
relationships. The choice of embedding model should consider your domain, language, and
performance requirements. Production systems often benchmark multiple embedding models
before selecting one."""

print(f"Document length: {len(sample_document)} characters")
print(f"Word count: {len(sample_document.split())} words")

## 2. Fixed-Size Chunking

Split text into chunks of exactly N characters, with optional overlap. Simple but can break mid-sentence.

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separator="\n"
)

fixed_chunks = fixed_splitter.split_text(sample_document)

print(f"Fixed-size: {len(fixed_chunks)} chunks\n")
for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:120] + ("..." if len(chunk) > 120 else ""))
    print()

Notice how fixed-size splitting cuts mid-sentence. This breaks context and hurts retrieval quality.

## 3. Recursive Character Splitting

Tries separators in priority order: paragraphs → sentences → words → characters. Respects natural text boundaries.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(sample_document)

print(f"Recursive: {len(recursive_chunks)} chunks\n")
for i, chunk in enumerate(recursive_chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:120] + ("..." if len(chunk) > 120 else ""))
    print()

Recursive splitting produces cleaner chunks that preserve sentence and paragraph boundaries.

## 4. Comparing Chunk Sizes

How does chunk size affect the number and quality of chunks?

In [ ]:
sizes = [100, 200, 400, 600, 1000]
overlap = 40

results = []
for size in sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(sample_document)
    avg_len = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    results.append({
        "chunk_size": size,
        "num_chunks": len(chunks),
        "avg_length": round(avg_len),
        "min_length": min(len(c) for c in chunks) if chunks else 0,
        "max_length": max(len(c) for c in chunks) if chunks else 0
    })

print(f"{'Size':>6} | {'Chunks':>6} | {'Avg Len':>7} | {'Min':>4} | {'Max':>4}")
print("-" * 45)
for r in results:
    print(f"{r['chunk_size']:>6} | {r['num_chunks']:>6} | {r['avg_length']:>7} | {r['min_length']:>4} | {r['max_length']:>4}")

## 5. Comparing Overlap Values

Overlap preserves context across chunk boundaries. Let's see how different overlap values affect chunk count.

In [ ]:
overlaps = [0, 20, 40, 80, 120]
chunk_size = 200

print(f"Chunk size: {chunk_size}\n")
for ov in overlaps:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=ov
    )
    chunks = splitter.split_text(sample_document)
    print(f"Overlap {ov:>3}: {len(chunks)} chunks")

More overlap = more chunks (more storage), but better context continuity. 10-20% of chunk size is a common default.

## 6. Visualizing Chunk Boundaries

See exactly where each strategy cuts the text.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def get_boundaries(text, chunks):
    """Find character positions where chunks start/end."""
    boundaries = []
    pos = 0
    for chunk in chunks:
        start = text.find(chunk[:50], pos)
        end = start + len(chunk)
        boundaries.append((start, end))
        pos = end
    return boundaries

# Generate chunks for comparison
fixed_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=40, separator="\n")
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)

fixed_c = fixed_splitter.split_text(sample_document)
recursive_c = recursive_splitter.split_text(sample_document)

fixed_bounds = get_boundaries(sample_document, fixed_c)
recursive_bounds = get_boundaries(sample_document, recursive_c)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 4))

for i, (bounds, title) in enumerate([
    (fixed_bounds, "Fixed-Size Chunking"),
    (recursive_bounds, "Recursive Chunking")
]):
    ax = axes[i]
    colors = plt.cm.Set3(np.linspace(0, 1, len(bounds)))
    for j, (start, end) in enumerate(bounds):
        ax.barh(0, end - start, left=start, height=0.5, color=colors[j], edgecolor="black", linewidth=0.5)
    ax.set_xlim(0, len(sample_document))
    ax.set_yticks([])
    ax.set_title(title)
    ax.set_xlabel("Character position")

plt.tight_layout()
plt.savefig("chunking_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_comparison.png")

## 7. Measuring Retrieval Quality

Build a small test set and measure which chunking strategy retrieves the most relevant chunks.

In [ ]:
from langchain_openai import ChatOpenAIEmbeddings
import numpy as np

embeddings = ChatOpenAIEmbeddings(model="models/text-embedding-3-small")

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieval_quality(chunks, test_queries, expected_keywords):
    """Measure what fraction of expected keywords appear in top retrieved chunks."""
    chunk_embeddings = embeddings.embed_documents(chunks)
    
    scores = []
    for query, keywords in zip(test_queries, expected_keywords):
        query_embedding = embeddings.embed_query(query)
        
        # Compute similarity with each chunk
        sims = [cosine_similarity(query_embedding, ce) for ce in chunk_embeddings]
        top_k_idx = np.argsort(sims)[-3:]  # top 3
        retrieved_text = " ".join(chunks[i] for i in top_k_idx)
        
        # Count how many expected keywords appear in retrieved text
        found = sum(1 for kw in keywords if kw.lower() in retrieved_text.lower())
        scores.append(found / len(keywords))
    
    return np.mean(scores)

test_queries = [
    "How does chunking affect RAG performance?",
    "What is the purpose of embedding overlap?",
    "How do vector databases store information?"
]

expected_keywords = [
    ["chunk", "large", "small", "context"],
    ["overlap", "continuity", "boundary"],
    ["vector", "embedding", "similarity", "search"]
]

strategies = {
    "Fixed-size": fixed_c,
    "Recursive": recursive_c
}

print("Retrieval Quality (keyword recall in top-3 chunks):\n")
for name, chunks in strategies.items():
    quality = retrieval_quality(chunks, test_queries, expected_keywords)
    print(f"  {name:>12}: {quality:.1%} keyword recall")

## 8. Semantic Chunking (Optional)

Semantic chunking groups sentences by meaning rather than size. Requires embedding during the split, so it's slower but produces more coherent chunks.

**Note:** This cell requires `langchain-experimental`.

In [ ]:
# Uncomment to run (requires langchain-experimental)
# !pip install langchain-experimental

# from langchain_experimental.text_splitter import SemanticChunker
#
# semantic_splitter = SemanticChunker(
#     embeddings,
#     breakpoint_threshold_type="percentile"
# )
# semantic_chunks = semantic_splitter.split_text(sample_document)
# print(f"Semantic: {len(semantic_chunks)} chunks\n")
# for i, chunk in enumerate(semantic_chunks):
#     print(f"--- Chunk {i} ({len(chunk)} chars) ---")
#     print(chunk[:120] + "...")
#     print()
#
# quality = retrieval_quality(semantic_chunks, test_queries, expected_keywords)
# print(f"Semantic chunking: {quality:.1%} keyword recall")

## Summary

| Strategy | Pros | Cons | Best For |
|---|---|---|---|
| Fixed-size | Simple, fast | Breaks mid-sentence | Quick prototypes |
| Recursive | Respects boundaries, good default | Slightly more code | Most use cases |
| Semantic | Best coherence | Slow, expensive | High-quality retrieval |

**Recommendation:** Start with `RecursiveCharacterTextSplitter` (chunk_size=500-1000, overlap=10-20%). Optimize from there.

Next: RAG optimization in `03-rag-optimization.ipynb`.